# 🚀 The Easy & Detailed NumPy Guide: From Zero to Machine Learning
> **A Comprehensive, Visual, and Intuitive Reference Guide to Vectorized Computing**
> 
> *Target Audience:* Beginners seeking crystal-clear clarity, Data Scientists looking for a thorough reference, and ML practitioners building fast algorithms without getting bogged down in dry mathematical proofs.

---

### 🌟 How to Use This Notebook
1. **Detailed Explanations in Plain English:** Every concept is explained step-by-step with clear real-world intuition, analogies, and "under-the-hood" mechanics—easy to read, not deeper or mathematically dense.
2. **Visual Mental Models:** Visual ASCII diagrams illustrate memory layouts, multidimensional slicing grids, axes directions, and broadcasting rules.
3. **Pre-Executed & Verified:** Every code cell is pre-executed with real outputs so you can read this like an illustrated technical handbook or run each cell interactively.
4. **Smooth Navigation:** Use the **Table of Contents** below or each chapter's `[⬆️ Back to Table of Contents]` link to jump directly to any section.

---

<a id="table-of-contents"></a>
## 📑 Table of Contents

| Part | Chapter | Key Topics Covered |
| :---: | :--- | :--- |
| **I** | [1. Why NumPy is Fast: The Core Mental Model](#ch1) | Python Lists vs ndarrays, Contiguous RAM, CPU Cache & SIMD, Vectorization |
| **I** | [2. Creating NumPy Arrays](#ch2) | `np.array`, `zeros`, `ones`, `full`, `arange` vs `linspace`, identity & diagonals |
| **I** | [3. Array Anatomy & Attributes](#ch3) | `shape`, `ndim`, `dtype`, `itemsize`, `nbytes`, `strides`, and memory flags |
| **I** | [4. Modern Random Number Generation](#ch4) | `default_rng()`, Uniform, Normal (Gaussian), Integers, Choice, Shuffling |
| **II** | [5. Indexing, Slicing & Striding](#ch5) | 1D & 2D slicing grids, negative indexing, step stride, ellipsis `...`, `newaxis` |
| **II** | [6. Views vs Copies: Memory Mechanics](#ch6) | Shared memory buffers, `arr.base`, accidental mutations, `.copy()` best practices |
| **II** | [7. Advanced Indexing & Filtering](#ch7) | Boolean masking, Bitwise operators (`&`, `\|`, `~`), `np.where`, Fancy indexing |
| **II** | [8. Reshaping & Structural Transforms](#ch8) | `reshape(-1)`, `flatten()` vs `ravel()`, Transpose `.T`, `squeeze`, `expand_dims` |
| **III** | [9. Combining, Stacking & Splitting Arrays](#ch9) | `concatenate`, `vstack`, `hstack`, `stack`, `split`, `tile` vs `repeat` |
| **III** | [10. Universal Functions (ufuncs)](#ch10) | Vectorized math, trigonometry, `reduce`, `accumulate`, `outer` operations |
| **III** | [11. Broadcasting Demystified: The 3 Rules](#ch11) | Shape compatibility rules, dimension stretching, feature normalization |
| **III** | [12. Aggregations & The Axis Model](#ch12) | Mental model for `axis=0` vs `axis=1`, `keepdims=True`, `nan`-safe functions |
| **IV** | [13. Sorting, Searching & Ranking](#ch13) | `sort` vs `argsort`, fast $O(N)$ top-K selection with `argpartition`, `unique` |
| **IV** | [14. Linear Algebra Made Intuitive](#ch14) | `@` vs `*`, Dot products, Matrix inverses, Solving $Ax=b$, Vector norms |
| **IV** | [15. Practical Machine Learning Vectorization](#ch15) | Pairwise Euclidean distance, Stable Softmax, One-Hot Encoding, MSE Loss |
| **IV** | [16. Performance Optimization & Anti-Patterns](#ch16) | Preallocation vs `np.append` anti-pattern, C-order vs Fortran cache lines, in-place math |
| **V** | [Quick Reference Cheat Sheet](#summary) | High-frequency function reference table and companion practice exercises |

<a id="ch1"></a>
---
# Part I: Foundations & Memory Architecture

## 1. Why NumPy is Fast: The Core Mental Model

Standard Python is dynamic, flexible, and beginner-friendly, but it is notoriously slow for numerical computation.
If you attempt to do scientific data processing, image manipulation, or machine learning using standard Python lists and `for` loops, your code will run **50x to 100x slower** than compiled languages like C or Fortran.

**NumPy (Numerical Python)** solves this performance gap. It provides a specialized, high-performance data structure called the **`ndarray`** (N-dimensional array).

---

### 1.1 Python Lists vs. NumPy ndarrays: The Memory Layout
To understand why NumPy is fundamentally faster, we have to look at how computer RAM stores data for both structures:

```
[Standard Python List]: Array of Pointers to Heap Objects
List Container ---> [ Pointer 0 ] ---> PyObject (type=int, refcount=1, value=42)
                    [ Pointer 1 ] ---> PyObject (type=int, refcount=1, value=18)
                    [ Pointer 2 ] ---> PyObject (type=int, refcount=1, value=99)
  * Values are scattered randomly across system RAM -> High CPU Cache Misses!
  * Type-checking and dynamic dispatch happen on EVERY single loop iteration.
  * Enormous memory overhead (approx 28+ bytes per simple integer!).

[NumPy ndarray]: One Contiguous C-Style Memory Buffer
Array Header ---> [ 42 | 18 | 99 | 74 | 12 | ... ] (raw bytes packed side-by-side)
  * Sequential in RAM -> Hardware CPU L1/L2 cache pre-fetching loads data ahead of time.
  * Fixed, uniform data type -> Direct pointer arithmetic: Memory Address = Base + Index * ItemSize.
  * Bypasses the Python interpreter during loops -> Dispatches directly to compiled SIMD CPU vector instructions.
```

### 1.2 What is Vectorization?
In standard Python, to add two lists element-by-element, you must write a `for` loop that unpacks each element, checks its type, computes the addition in Python bytecode, and packs the result into a new Python object.
In NumPy, **Vectorization** allows you to express batch operations with simple syntax like `c = a + b`.
Under the hood, NumPy delegates the entire loop down to pre-compiled C loops that use **SIMD (Single Instruction, Multiple Data)** instructions on your CPU processor. A single CPU instruction adds 4, 8, or 16 numbers simultaneously!

In [1]:
import numpy as np
import sys
import time

print(f"NumPy Version: {np.__version__}")

NumPy Version: 2.5.3


### 1.3 Memory Footprint Benchmark: List vs ndarray
Let's measure the actual memory consumed by a Python list of 10,000 integers versus a NumPy array of 10,000 integers.
Notice how much leaner and compact NumPy is in RAM:

In [2]:
# Create 10,000 integers in Python list vs NumPy array
n_items = 10_000
py_list = list(range(n_items))
np_arr = np.arange(n_items, dtype=np.int64)

# Memory of the list container + memory of each individual integer PyObject
py_list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list)
# Memory of the NumPy array buffer + array header
np_arr_bytes = np_arr.nbytes + sys.getsizeof(np_arr)

print(f"📦 Python List Memory:   {py_list_bytes:>8,} bytes")
print(f"⚡ NumPy Array Memory:    {np_arr_bytes:>8,} bytes")
print(f"🎉 NumPy is {py_list_bytes / np_arr_bytes:.1f}x more memory-compact!")

📦 Python List Memory:    360,056 bytes
⚡ NumPy Array Memory:     160,112 bytes
🎉 NumPy is 2.2x more memory-compact!


### 1.4 Execution Speed Benchmark: Vectorization vs Python Loop
Now let's compute element-wise addition on two arrays of **5,000,000 numbers**.
Notice the massive speedup when running NumPy's compiled C code versus interpreted Python loops:

In [3]:
N = 5_000_000
list_a = list(range(N))
list_b = list(range(N))
arr_a = np.arange(N, dtype=np.float64)
arr_b = np.arange(N, dtype=np.float64)

# 1. Pure Python Loop (benchmarked on 500k and scaled for timing responsiveness)
t0 = time.perf_counter()
_ = [a + b for a, b in zip(list_a[:500_000], list_b[:500_000])]
t1 = time.perf_counter()
py_time = (t1 - t0) * 10 # Extrapolated to 5M

# 2. NumPy Vectorized C-Speed Addition
t0 = time.perf_counter()
_ = arr_a + arr_b
t1 = time.perf_counter()
np_time = t1 - t0

print(f"🐢 Python Loop (extrapolated 5M): {py_time:.4f} seconds")
print(f"🚀 NumPy Vectorized (5M):         {np_time:.4f} seconds")
print(f"⚡ Speedup:                       {py_time / np_time:.1f}x faster!")

🐢 Python Loop (extrapolated 5M): 0.3458 seconds
🚀 NumPy Vectorized (5M):         0.0199 seconds
⚡ Speedup:                       17.4x faster!


> 💡 **Key Takeaway:**
> Never write `for` loops in Python to do math on arrays. NumPy's vectorization executes directly on CPU vector hardware at native C speed!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch2"></a>
---
## 2. Creating NumPy Arrays

NumPy provides built-in constructors to create arrays of any dimension, shape, and data type without writing loops.

### 2.1 From Python Sequences (`np.array`)
You can convert any standard Python list or nested list into an `ndarray`.
Specifying the `dtype` explicitly prevents unintended automatic type conversions.

In [4]:
# 1D Array (Vector) with explicit 32-bit float
a_1d = np.array([10, 20, 30, 40], dtype=np.float32)

# 2D Matrix (2 rows x 3 columns)
a_2d = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int32)

print("1D Array (float32):", a_1d)
print("\n2D Matrix (2x3):\n", a_2d)

1D Array (float32): [10. 20. 30. 40.]

2D Matrix (2x3):
 [[1 2 3]
 [4 5 6]]


### 2.2 Constant Initializers: Zeros, Ones, Full, Empty
These initializers pre-allocate memory buffers before running computations (e.g. allocating arrays for model weights, image masks, or accumulators):
- **`np.zeros(shape)`**: Fills all elements with `0`.
- **`np.ones(shape)`**: Fills all elements with `1`.
- **`np.full(shape, fill_value)`**: Fills all elements with a chosen constant value.
- **`np.empty(shape)`**: Allocates memory without clearing it (fastest, but contains whatever random junk bytes were in RAM!).

In [5]:
# Zeros array of shape (2, 4)
zeros = np.zeros(shape=(2, 4), dtype=np.float64)

# Ones array of shape (2, 3)
ones = np.ones(shape=(2, 3), dtype=np.int16)

# Full array with a custom scalar (e.g., 99.0)
filled = np.full(shape=(2, 3), fill_value=99.0)

# Empty array (uninitialized buffer)
empty = np.empty(shape=(2, 2))

print("Zeros (2x4):\n", zeros)
print("\nOnes (2x3):\n", ones)
print("\nFull with 99.0 (2x3):\n", filled)

Zeros (2x4):
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Ones (2x3):
 [[1 1 1]
 [1 1 1]]

Full with 99.0 (2x3):
 [[99. 99. 99.]
 [99. 99. 99.]]


### 2.3 Like-Constructors: Matching Shape & Dtype
When you already have an existing array and want another array with the identical shape and data type without manually typing out the dimensions:
- `np.zeros_like(template)`
- `np.ones_like(template)`
- `np.full_like(template, fill_value)`

In [6]:
template = np.array([[1.5, 2.5, 3.5], [4.5, 5.5, 6.5]])

zeros_like = np.zeros_like(template)
full_like = np.full_like(template, fill_value=-1.0)

print("Template shape:", template.shape, "dtype:", template.dtype)
print("\nZeros like template:\n", zeros_like)
print("\nFull like template:\n", full_like)

Template shape: (2, 3) dtype: float64

Zeros like template:
 [[0. 0. 0.]
 [0. 0. 0.]]

Full like template:
 [[-1. -1. -1.]
 [-1. -1. -1.]]


### 2.4 Numerical Ranges: `arange` vs `linspace`
This is one of the most critical distinctions in scientific Python:

| Function | Signature | Use When... | End-point Included? |
| :--- | :--- | :--- | :---: |
| `np.arange(start, stop, step)` | Specify **step size** | You care about the step interval between elements | ❌ No (stop is exclusive) |
| `np.linspace(start, stop, num)` | Specify **number of points** | You care about exact sample count | ✅ Yes (stop is inclusive) |

> ⚠️ **Pro-Tip on Floating Point:**
> Avoid using `np.arange` with floating-point step sizes like `np.arange(0, 1, 0.1)` because floating-point rounding errors can result in unexpected array lengths. Always prefer `np.linspace(0, 1, 11)` for float ranges!

In [7]:
# arange: step is 2 (stop is exclusive)
r_arange = np.arange(start=0, stop=10, step=2)

# linspace: exactly 5 points evenly spaced between 0 and 1 (inclusive)
r_linspace = np.linspace(start=0.0, stop=1.0, num=5)

# logspace: 4 points logarithmically spaced from 10^1 to 10^4
r_logspace = np.logspace(start=1, stop=4, num=4, base=10.0)

print("np.arange(0, 10, 2):     ", r_arange)
print("np.linspace(0, 1, num=5):", r_linspace)
print("np.logspace(1, 4, num=4):", r_logspace)

np.arange(0, 10, 2):      [0 2 4 6 8]
np.linspace(0, 1, num=5): [0.   0.25 0.5  0.75 1.  ]
np.logspace(1, 4, num=4): [   10.   100.  1000. 10000.]


### 2.5 Identity & Diagonal Matrices
Essential for linear transformations, baseline weights, and one-hot encoding:
- **`np.eye(N)`**: Creates an $N \times N$ identity matrix with 1s on the main diagonal and 0s elsewhere.
- **`np.diag(v)`**: If $v$ is a 1D vector, creates a 2D matrix with $v$ along the diagonal. If $v$ is a 2D matrix, extracts the diagonal elements.

In [8]:
# 3x3 Identity matrix
identity = np.eye(3)

# Construct a diagonal matrix from a list of diagonal values
diag_matrix = np.diag([10, 20, 30])

# Upper and Lower triangular matrices
mat = np.ones((3, 3), dtype=int)
upper = np.triu(mat)
lower = np.tril(mat)

print("Identity Matrix (3x3):\n", identity)
print("\nDiagonal Matrix:\n", diag_matrix)
print("\nUpper Triangular Matrix:\n", upper)

Identity Matrix (3x3):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Diagonal Matrix:
 [[10  0  0]
 [ 0 20  0]
 [ 0  0 30]]

Upper Triangular Matrix:
 [[1 1 1]
 [0 1 1]
 [0 0 1]]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch3"></a>
---
## 3. Array Anatomy, Attributes & Memory

Every NumPy array consists of two components:
1. **The Data Buffer**: Raw bytes stored in contiguous system RAM.
2. **The Array Header (Metadata)**: A lightweight Python object storing structural descriptors: `shape`, `ndim`, `dtype`, `itemsize`, `nbytes`, and `strides`.

```
                    ARRAY HEADER (Metadata in Python)
               shape   : (3, 4)      -> 3 rows, 4 columns
               ndim    : 2           -> 2 dimensions
               dtype   : int32       -> 4 bytes per integer
               itemsize: 4           -> size of 1 item
               nbytes  : 48          -> total bytes (12 * 4)
               strides : (16, 4)     -> jump 16 bytes for next row, 4 for next col
                  |
                  v
               DATA BUFFER (Raw contiguous bytes in RAM)
               [ 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 ]
```

In [9]:
# Create a 3x4 array of 32-bit integers
arr = np.arange(12, dtype=np.int32).reshape(3, 4)

print("--- Array Structural Attributes ---")
print("Array:\n", arr)
print(f"ndim (dimensions):       {arr.ndim}")
print(f"shape (rows, cols):      {arr.shape}")
print(f"size (total elements):   {arr.size}")
print(f"dtype (data type):       {arr.dtype}")
print(f"itemsize (bytes/item):   {arr.itemsize} bytes")
print(f"nbytes (total memory):   {arr.nbytes} bytes ({arr.size} * {arr.itemsize})")

--- Array Structural Attributes ---
Array:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
ndim (dimensions):       2
shape (rows, cols):      (3, 4)
size (total elements):   12
dtype (data type):       int32
itemsize (bytes/item):   4 bytes
nbytes (total memory):   48 bytes (12 * 4)


### 3.2 Demystifying `strides`
**Strides** is one of NumPy's most brilliant features. It tells the CPU how many bytes it must step forward in RAM to advance by 1 index in each dimension:
- To advance to the next column: step **4 bytes** (the size of one 32-bit integer).
- To advance to the next row: step 4 columns $\times$ 4 bytes = **16 bytes**.

```
Row 0: [ 0 (byte 0)  |  1 (byte 4)  |  2 (byte 8)  |  3 (byte 12) ]
Row 1: [ 4 (byte 16) |  5 (byte 20) |  6 (byte 24) |  7 (byte 28) ]
Row 2: [ 8 (byte 32) |  9 (byte 36) | 10 (byte 40) | 11 (byte 44) ]
```

> 🧠 **Why Strides Matter (Under the Hood):**
> When you transpose an array (`arr.T`), NumPy **does not copy a single byte of data**!
> Instead, it simply swaps the strides tuple from `(16, 4)` to `(4, 16)`. The CPU navigates the exact same memory buffer in reverse order. This is why transposing a 10 GB matrix in NumPy executes in $0.000001$ seconds!

In [10]:
print("Strides tuple (row_bytes, col_bytes):", arr.strides)

# Inspect memory flags: is it C-contiguous? Does it own its memory?
print("\nMemory Flags:")
print("C-Contiguous (Row-major):", arr.flags['C_CONTIGUOUS'])
print("Fortran (Column-major):  ", arr.flags['F_CONTIGUOUS'])
print("Owns its memory buffer:  ", arr.flags['OWNDATA'])

Strides tuple (row_bytes, col_bytes): (16, 4)

Memory Flags:
C-Contiguous (Row-major): True
Fortran (Column-major):   False
Owns its memory buffer:   False


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch4"></a>
---
## 4. Modern Random Number Generation (`np.random.default_rng`)

In older NumPy tutorials, you will see `np.random.seed(42)` and `np.random.rand()`.
**NumPy 1.17+ officially deprecated that legacy API.**
The modern standard is the **Generator API** initialized via `np.random.default_rng()`, which uses the high-performance **PCG64** pseudo-random number generator algorithm. It is faster, statistically superior, and thread-safe.

### 4.1 Common Distributions
- **`rng.random(size)`**: Uniform floats in $[0.0, 1.0)$.
- **`rng.integers(low, high, size)`**: Uniform integers in $[\text{low}, \text{high})$.
- **`rng.standard_normal(size)`**: Standard Normal (Gaussian) distribution ($\mu=0, \sigma=1$).
- **`rng.normal(loc, scale, size)`**: Gaussian with custom mean (`loc`) and standard deviation (`scale`).

In [11]:
# Initialize modern generator with a fixed seed for reproducibility
rng = np.random.default_rng(seed=42)

# 1. Random floats in [0.0, 1.0)
floats = rng.random(size=(2, 3))

# 2. Random integers between 1 and 100
ints = rng.integers(low=1, high=100, size=(2, 3))

# 3. Standard Normal distribution (mean=0, std=1)
std_norm = rng.standard_normal(size=(2, 3))

# 4. Custom Normal distribution (mean=10.0, std=2.5)
custom_norm = rng.normal(loc=10.0, scale=2.5, size=4)

print("Floats [0, 1):\n", floats.round(3))
print("\nIntegers [1, 100):\n", ints)
print("\nStandard Normal (Gaussian):\n", std_norm.round(3))
print("\nNormal(mu=10, sigma=2.5):\n", custom_norm.round(2))

Floats [0, 1):
 [[0.774 0.439 0.859]
 [0.697 0.094 0.976]]

Integers [1, 100):
 [[73 76 72]
 [78 51 13]]

Standard Normal (Gaussian):
 [[-0.853  0.879  0.778]
 [ 0.066  1.127  0.468]]

Normal(mu=10, sigma=2.5):
 [ 7.85 10.92  7.6  12.2 ]


### 4.2 Shuffling and Random Sampling
Used constantly in machine learning for creating train/validation splits, mini-batch shuffling, and bootstrapping:
- **`rng.choice(a, size, replace, p)`**: Sample elements with or without replacement.
- **`rng.shuffle(arr)`**: Shuffles the array **in-place** along the first axis.
- **`rng.permutation(arr)`**: Returns a **new shuffled copy** without mutating the original.

In [12]:
items = np.array(['Apple', 'Banana', 'Cherry', 'Date', 'Elderberry'])

# Randomly sample 3 items with replacement
sample_replace = rng.choice(items, size=3, replace=True)

# Randomly sample 3 items without replacement (all unique)
sample_unique = rng.choice(items, size=3, replace=False)

# Random permutation (creates a shuffled copy)
shuffled_items = rng.permutation(items)

print("Sample with replacement:   ", sample_replace)
print("Sample without replacement:", sample_unique)
print("Shuffled copy:             ", shuffled_items)
print("Original unchanged:        ", items)

Sample with replacement:    ['Banana' 'Date' 'Apple']
Sample without replacement: ['Banana' 'Date' 'Cherry']
Shuffled copy:              ['Cherry' 'Banana' 'Apple' 'Date' 'Elderberry']
Original unchanged:         ['Apple' 'Banana' 'Cherry' 'Date' 'Elderberry']


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch5"></a>
---
# Part II: Slicing, Indexing & Views

## 5. Indexing, Slicing & Striding

Indexing allows you to extract individual elements, sub-matrices, or multidimensional slices.
NumPy slicing follows standard Python syntax: `start:stop:step`, where:
- `start` is inclusive (default is 0).
- `stop` is exclusive (default is end).
- `step` is the stride interval (can be negative to reverse!).

### 5.1 1D Array Slicing

In [13]:
a = np.array([10, 20, 30, 40, 50, 60, 70, 80])

print("Original:             ", a)
print("Slice [2:6]:          ", a[2:6])      # Elements from index 2 to 5
print("Every 2nd item [::2]: ", a[::2])      # Stride of 2
print("Reverse array [::-1]: ", a[::-1])     # Negative step reverses array!

Original:              [10 20 30 40 50 60 70 80]
Slice [2:6]:           [30 40 50 60]
Every 2nd item [::2]:  [10 30 50 70]
Reverse array [::-1]:  [80 70 60 50 40 30 20 10]


### 5.2 2D Matrix Indexing & Slicing: `[row_slice, col_slice]`
In standard Python lists, 2D access requires chained brackets `list_2d[row][col]`.
In NumPy, you use a single pair of brackets with a comma separating dimensions: **`matrix[row_slice, col_slice]`**.

```
                Matrix shape (4, 5)
               Col 0   Col 1   Col 2   Col 3   Col 4
             +-------+-------+-------+-------+-------+
    Row 0    |   0   |   1   |   2   |   3   |   4   |
             +-------+-------+-------+-------+-------+
    Row 1    |  10   |  11   |  12   |  13   |  14   |  <- Row 1:3, Col 1:4
             +-------+-------+-------+-------+-------+     extracts the subgrid
    Row 2    |  20   |  21   |  22   |  23   |  24   |     [[11, 12, 13],
             +-------+-------+-------+-------+-------+      [21, 22, 23]]
    Row 3    |  30   |  31   |  32   |  33   |  34   |
             +-------+-------+-------+-------+-------+
```

In [14]:
grid = np.arange(20).reshape(4, 5)
print("Full 4x5 Grid:\n", grid)

# 1. Single scalar element: Row 1, Column 3
print("\nElement at (row 1, col 3):", grid[1, 3])

# 2. Entire Row 2: (use ':' for all columns)
print("\nRow 2 (all columns):", grid[2, :])

# 3. Entire Column 1: (all rows)
print("\nColumn 1 (all rows):", grid[:, 1])

# 4. Sub-matrix: Rows 1 to 3, Columns 1 to 4
print("\nSubgrid [1:3, 1:4]:\n", grid[1:3, 1:4])

Full 4x5 Grid:
 [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

Element at (row 1, col 3): 8

Row 2 (all columns): [10 11 12 13 14]

Column 1 (all rows): [ 1  6 11 16]

Subgrid [1:3, 1:4]:
 [[ 6  7  8]
 [11 12 13]]


### 5.3 3D Tensor Slicing & The Ellipsis (`...`)
When working with higher-dimensional tensors (e.g., images of shape `(batch, height, width, channels)`), typing multiple colons `[:, :, :, 0]` gets tedious.
The **ellipsis (`...`)** automatically expands to cover as many full `:` slices as needed!
`tensor[..., 0]` means: select the first channel across all preceding dimensions.

In [15]:
# 3D Tensor of shape (batch=2, rows=3, cols=4)
tensor = np.arange(24).reshape(2, 3, 4)

print("Tensor Shape:", tensor.shape)
# Extract the first slice along the last dimension using ellipsis
last_dim_first_col = tensor[..., 0]
print("Slice tensor[..., 0] shape:", last_dim_first_col.shape)
print("Values:\n", last_dim_first_col)

Tensor Shape: (2, 3, 4)
Slice tensor[..., 0] shape: (2, 3)
Values:
 [[ 0  4  8]
 [12 16 20]]


### 5.4 Adding Dimensions: `np.newaxis` / `None`
Often you need to convert a 1D vector of shape `(N,)` into a 2D column vector `(N, 1)` or row vector `(1, N)` for matrix multiplication and broadcasting.
Inserting `np.newaxis` (or `None`) creates a new axis of length 1:

In [16]:
vec = np.array([1, 2, 3, 4])
print("Original 1D Vector shape:", vec.shape)

# Convert to 2D Row Vector: shape (1, 4)
row_vec = vec[np.newaxis, :]
print("Row Vector shape (1, 4):  ", row_vec.shape)

# Convert to 2D Column Vector: shape (4, 1)
col_vec = vec[:, np.newaxis]
print("Column Vector shape (4, 1):", col_vec.shape)
print("Column Vector values:\n", col_vec)

Original 1D Vector shape: (4,)
Row Vector shape (1, 4):   (1, 4)
Column Vector shape (4, 1): (4, 1)
Column Vector values:
 [[1]
 [2]
 [3]
 [4]]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch6"></a>
---
## 6. Views vs Copies: Memory Mechanics

This is the **#1 source of silent bugs** for Python programmers transitioning to NumPy.

- **In Standard Python:** Slicing a list `sub = my_list[0:3]` creates a **copy**. Modifying `sub` does NOT alter `my_list`.
- **In NumPy:** Slicing an array `sub = my_arr[0:3]` returns a **VIEW**! It shares the exact same memory buffer. Modifying `sub` **MUTATES THE ORIGINAL ARRAY**!

### 6.1 Accidental Mutation via Views
Let's see this in action:

In [17]:
original = np.array([10, 20, 30, 40, 50])

# Slice returns a VIEW, not a copy!
view_slice = original[1:4]
print("Before mutation:")
print("Original:  ", original)
print("View Slice:", view_slice)

# Modify the view
view_slice[0] = 999

print("\nAfter mutating view_slice[0] = 999:")
print("View Slice:", view_slice)
print("Original:  ", original) # ORIGINAL HAS BEEN MUTATED!

Before mutation:
Original:   [10 20 30 40 50]
View Slice: [20 30 40]

After mutating view_slice[0] = 999:
View Slice: [999  30  40]
Original:   [ 10 999  30  40  50]


### 6.2 Checking Memory Ownership with `.base`
How can you programmatically verify whether an array owns its memory or is merely a view into another array?
Check the **`.base`** attribute:
- If `arr.base is None`: The array **owns its memory** (it is a standalone copy).
- If `arr.base is not None`: The array is a **view** pointing to the parent buffer stored in `.base`.

In [18]:
# Check .base
print("view_slice.base is original?", view_slice.base is original)

# Creating an explicit, safe copy
safe_copy = original[1:4].copy()
safe_copy[0] = -777

print("\nsafe_copy.base is None?", safe_copy.base is None)
print("Safe Copy:", safe_copy)
print("Original remains untouched:", original)

view_slice.base is original? True

safe_copy.base is None? True
Safe Copy: [-777   30   40]
Original remains untouched: [ 10 999  30  40  50]


> 💡 **The Golden Rule:**
> Basic slicing (`arr[1:4]`, `arr[:, 2]`) always returns a **view** for maximum speed.
> If you want an independent object, always explicitly call **`.copy()`**!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch7"></a>
---
## 7. Advanced Indexing & Filtering

Advanced indexing allows you to query, filter, and extract elements based on conditions or arbitrary index coordinate lists.

### 7.1 Boolean Masking
You can pass a boolean condition directly inside brackets: `arr[condition]`.
NumPy evaluates the condition element-wise and returns a 1D array of only the elements where the condition is `True`.

In [19]:
data = np.array([12, -5, 8, 0, -19, 25, 4, -2])

# Create a boolean condition
is_positive = data > 0
print("Boolean Mask:\n", is_positive)

# Filter values using the mask
positive_values = data[is_positive]
print("\nPositive values only:\n", positive_values)

Boolean Mask:
 [ True False  True False False  True  True False]

Positive values only:
 [12  8 25  4]


### 7.2 Combining Conditions: Bitwise `&`, `|`, `~`
Just like in Pandas, you **cannot** use Python's `and`, `or`, and `not` on NumPy arrays. You must use:
- **`&`** for AND
- **`|`** for OR
- **`~`** for NOT
- **Parentheses `()` around every condition** are mandatory!

In [20]:
# Find numbers that are positive AND even
pos_and_even = data[(data > 0) & (data % 2 == 0)]

# Find numbers that are negative OR greater than 20
neg_or_large = data[(data < 0) | (data > 20)]

print("Positive AND Even:       ", pos_and_even)
print("Negative OR Greater than 20:", neg_or_large)

Positive AND Even:        [12  8  4]
Negative OR Greater than 20: [ -5 -19  25  -2]


### 7.3 Vectorized Conditional Choice: `np.where`
`np.where(condition, value_if_true, value_if_false)` works like Excel's `IF()` or SQL's `CASE WHEN`:

In [21]:
scores = np.array([85, 42, 91, 68, 74])

# If score >= 70 return 'Pass', else return 'Fail'
status = np.where(scores >= 70, 'Pass', 'Fail')

# Clamp negative values to zero (ReLU activation logic!)
clamped = np.where(data > 0, data, 0)

print("Scores: ", scores)
print("Status: ", status)
print("\nOriginal Data: ", data)
print("Clamped (>=0):  ", clamped)

Scores:  [85 42 91 68 74]
Status:  ['Pass' 'Fail' 'Pass' 'Fail' 'Pass']

Original Data:  [ 12  -5   8   0 -19  25   4  -2]
Clamped (>=0):   [12  0  8  0  0 25  4  0]


### 7.4 Fancy (Integer Array) Indexing
Fancy indexing means passing a list or array of integer indices: `arr[[0, 3, 4]]`.
**Important Rule:** Unlike slicing which produces a view, **fancy indexing always returns a brand new COPY**!

In [22]:
arr_pool = np.array([100, 200, 300, 400, 500, 600])

# Select indices 0, 2, and 5
selected = arr_pool[[0, 2, 5]]
print("Selected elements via fancy indexing:", selected)

# Modifying selected does NOT mutate arr_pool because it is a copy!
selected[0] = 9999
print("Original pool remains unchanged:", arr_pool)

Selected elements via fancy indexing: [100 300 600]
Original pool remains unchanged: [100 200 300 400 500 600]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch8"></a>
---
## 8. Reshaping & Structural Transforms

Reshaping changes the dimensions of an array without modifying its data values.

### 8.1 The `.reshape()` Method & `-1` Auto-Inference
When reshaping, the total number of elements must stay identical (`rows * cols == total elements`).
NumPy allows you to pass `-1` for **one dimension**, and it will automatically calculate the correct size!

In [23]:
a = np.arange(12)
print("Original 1D shape:", a.shape)

# Reshape into 3 rows and 4 columns
matrix_3x4 = a.reshape(3, 4)
print("\nReshaped to (3, 4):\n", matrix_3x4)

# Use -1 to let NumPy infer columns: 2 rows -> auto-computes 6 columns
matrix_2x6 = a.reshape(2, -1)
print("\nReshaped to (2, -1) -> inferred shape:", matrix_2x6.shape)

Original 1D shape: (12,)

Reshaped to (3, 4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Reshaped to (2, -1) -> inferred shape: (2, 6)


### 8.2 Flattening: `flatten()` vs `ravel()`
Both collapse a multi-dimensional array into a 1D vector, but their memory behavior is different:
- **`arr.ravel()`**: Returns a **view** whenever possible (zero memory overhead, ultra fast).
- **`arr.flatten()`**: Always allocates and returns a brand-new **copy** in memory.

In [24]:
mat = np.array([[1, 2], [3, 4]])

flat_view = mat.ravel()      # View
flat_copy = mat.flatten()    # Copy

print("flat_view.base is mat? ", flat_view.base is mat)
print("flat_copy.base is None?", flat_copy.base is None)

flat_view.base is mat?  True
flat_copy.base is None? True


### 8.3 Transpose (`.T`) & Squeezing Extra Dimensions
- **`.T`**: Swaps axes (rows become columns).
- **`np.squeeze(arr)`**: Drops any redundant dimensions of length 1 (e.g. transforms shape `(1, 4, 1)` into `(4,)`).

In [25]:
# Transpose
print("Matrix:\n", matrix_3x4)
print("\nTransposed (4x3):\n", matrix_3x4.T)

# Squeeze redundant dimensions
redundant = np.zeros((1, 5, 1))
print("\nBefore squeeze shape:", redundant.shape)
squeezed = np.squeeze(redundant)
print("After squeeze shape: ", squeezed.shape)

Matrix:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Transposed (4x3):
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]

Before squeeze shape: (1, 5, 1)
After squeeze shape:  (5,)


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch9"></a>
---
# Part III: Mathematical Operations & Broadcasting

## 9. Combining, Stacking & Splitting Arrays

### 9.1 Stacking Arrays: `vstack`, `hstack`, `concatenate`
- **`np.vstack([a, b])`**: Stacks arrays **vertically** (row on top of row; increases rows).
- **`np.hstack([a, b])`**: Stacks arrays **horizontally** (side-by-side; increases columns).
- **`np.concatenate([a, b], axis=0)`**: General concatenation along any specified axis.

In [26]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

# Vertical stack
v_stacked = np.vstack([a, b])

# Horizontal stack
h_stacked = np.hstack([a, b])

print("Vertical Stack (rows added):\n", v_stacked)
print("\nHorizontal Stack (columns added):\n", h_stacked)

Vertical Stack (rows added):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]

Horizontal Stack (columns added):
 [[1 2 5 6]
 [3 4 7 8]]


### 9.2 Splitting Arrays
Use `np.split(arr, indices_or_sections, axis=...)` to cut an array into sub-arrays:

In [27]:
big_arr = np.arange(12)

# Split into 3 equal portions
parts = np.split(big_arr, 3)
for idx, part in enumerate(parts):
    print(f"Part {idx}: {part}")

Part 0: [0 1 2 3]
Part 1: [4 5 6 7]
Part 2: [ 8  9 10 11]


### 9.3 Tiling vs Repeating
- **`np.repeat(arr, repeats)`**: Duplicates individual elements consecutively `[1, 1, 2, 2]`.
- **`np.tile(arr, reps)`**: Tiles the entire block like floor tiles `[1, 2, 1, 2]`.

In [28]:
base = np.array([1, 2, 3])

repeated = np.repeat(base, 2)
tiled = np.tile(base, 2)

print("Base:    ", base)
print("Repeated:", repeated)
print("Tiled:   ", tiled)

Base:     [1 2 3]
Repeated: [1 1 2 2 3 3]
Tiled:    [1 2 3 1 2 3]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch10"></a>
---
## 10. Universal Functions (ufuncs)

A **universal function (ufunc)** is a function that operates on ndarrays element-by-element at compiled C speed.

### 10.1 Element-Wise Arithmetic & Math Functions
All standard operators (`+`, `-`, `*`, `/`, `**`, `//`, `%`) map directly to optimized ufuncs:

In [29]:
x = np.array([1.0, 4.0, 9.0, 16.0])

print("Square root: ", np.sqrt(x))
print("Exponential: ", np.exp(np.array([1, 2, 3])))
print("Natural log: ", np.log(x))
print("Sine:        ", np.sin(np.array([0, np.pi/2, np.pi])).round(2))

Square root:  [1. 2. 3. 4.]
Exponential:  [ 2.71828183  7.3890561  20.08553692]
Natural log:  [0.         1.38629436 2.19722458 2.77258872]
Sine:         [0. 1. 0.]


### 10.2 Special ufunc Methods: `reduce`, `accumulate`, `outer`
Built-in ufuncs have superpower methods:
- **`.reduce()`**: Applies the operation sequentially across the array (e.g. `np.add.reduce(arr)` is equivalent to `arr.sum()`).
- **`.accumulate()`**: Keeps running running totals (e.g. `np.add.accumulate(arr)` is `cumsum()`).
- **`.outer()`**: Computes the pairwise operation between all pairs of two vectors to create a matrix (e.g. a multiplication table!).

In [30]:
nums = np.array([1, 2, 3, 4, 5])

# Running cumulative sum
running_sum = np.add.accumulate(nums)
print("Running cumulative sum:", running_sum)

# Outer multiplication table (1 to 5 times 1 to 5)
mult_table = np.multiply.outer(nums, nums)
print("\n5x5 Multiplication Table:\n", mult_table)

Running cumulative sum: [ 1  3  6 10 15]

5x5 Multiplication Table:
 [[ 1  2  3  4  5]
 [ 2  4  6  8 10]
 [ 3  6  9 12 15]
 [ 4  8 12 16 20]
 [ 5 10 15 20 25]]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch11"></a>
---
## 11. Broadcasting Demystified: The 3 Rules

Broadcasting describes how NumPy treats arrays with different shapes during arithmetic operations without making unnecessary copies of data in RAM.

### 11.1 The 3 Broadcasting Rules
When operating on two arrays, NumPy compares their shapes **element-wise from right to left** (trailing dimensions first):
1. **Rule 1:** If the two arrays have a different number of dimensions, pad the shape of the smaller array with `1`s on its **left** side.
2. **Rule 2:** Two dimensions are **compatible** if they are equal, or if one of them is `1`.
3. **Rule 3:** If a dimension has size `1`, that dimension is virtually stretched to match the larger dimension!

```
                  VISUAL EXAMPLE OF BROADCASTING
                  
   Matrix A: shape (3, 3)             Vector B: shape (3,)
   +----+----+----+                   Padded to: shape (1, 3)
   | 10 | 10 | 10 |                     +----+----+----+
   +----+----+----+                     |  1 |  2 |  3 |
   | 20 | 20 | 20 |                     +----+----+----+
   +----+----+----+                            |
   | 30 | 30 | 30 |                 Virtually stretched down
   +----+----+----+                   +----+----+----+
          +                           |  1 |  2 |  3 |
                                      +----+----+----+
                                      |  1 |  2 |  3 |
                                      +----+----+----+
                                      |  1 |  2 |  3 |
                                      +----+----+----+
                                             =
                                      +----+----+----+
                                      | 11 | 12 | 13 |
                                      +----+----+----+
                                      | 21 | 22 | 23 |
                                      +----+----+----+
                                      | 31 | 32 | 33 |
                                      +----+----+----+
```

In [31]:
# Matrix shape (3, 3)
A = np.array([[10, 10, 10], [20, 20, 20], [30, 30, 30]])

# 1D Vector shape (3,) -> broadcasts seamlessly across rows!
B = np.array([1, 2, 3])

result = A + B
print("A (3x3) + B (3,) =\n", result)

A (3x3) + B (3,) =
 [[11 12 13]
 [21 22 23]
 [31 32 33]]


### 11.2 Practical Example: Feature Standardization (Z-score Normalization)
In Machine Learning, we standardize features by subtracting the column mean and dividing by the column standard deviation:
$$Z = \frac{X - \mu}{\sigma}$$
Broadcasting performs this entire matrix calculation in one clean line!

In [32]:
# 4 samples with 3 features each: shape (4, 3)
X = np.array([
    [10.0, 200.0, 1.5],
    [12.0, 240.0, 1.8],
    [9.0,  180.0, 1.2],
    [11.0, 210.0, 1.6]
])

# Mean and std computed along rows (axis=0): shape (3,)
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)

# Broadcasted Z-score standardization!
X_standardized = (X - mean) / std

print("Feature Means:    ", mean.round(2))
print("Feature Std Devs: ", std.round(2))
print("\nStandardized Matrix (Zero mean, unit variance):\n", X_standardized.round(2))

Feature Means:    

 [ 10.5  207.5    1.52]
Feature Std Devs:  [ 1.12 21.65  0.22]

Standardized Matrix (Zero mean, unit variance):
 [[-0.45 -0.35 -0.12]
 [ 1.34  1.5   1.27]
 [-1.34 -1.27 -1.5 ]
 [ 0.45  0.12  0.35]]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch12"></a>
---
## 12. Aggregations & The Axis Model

Aggregations collapse multi-dimensional data into summary statistics (e.g. `sum`, `mean`, `std`, `min`, `max`).

### 12.1 The Axis Direction Mental Model
Beginners frequently confuse `axis=0` and `axis=1`. Here is the easiest way to remember:
- **`axis=0`**: Collapse **downward across rows** (computes column-wise summaries).
- **`axis=1`**: Collapse **horizontally across columns** (computes row-wise summaries).

```
                      axis=1 (Horizontal collapse -> row summaries)
                             -------->
                    Col 0   Col 1   Col 2
                  +-------+-------+-------+
  axis=0   Row 0  |   1   |   2   |   3   |  -> sum(Row 0) = 6
(Vertical  Row 1  |   4   |   5   |   6   |  -> sum(Row 1) = 15
collapse)  Row 2  |   7   |   8   |   9   |  -> sum(Row 2) = 24
    |             +-------+-------+-------+
    v                 |       |       |
  sum(Cols) =         12      15      18
```

In [33]:
mat = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

print("Overall Matrix Sum:             ", np.sum(mat))
print("Column-wise Sums (axis=0):      ", np.sum(mat, axis=0))
print("Row-wise Sums (axis=1):         ", np.sum(mat, axis=1))
print("Row-wise Max Values (axis=1):   ", np.max(mat, axis=1))
print("Index of Max in each row:       ", np.argmax(mat, axis=1))

Overall Matrix Sum:              45
Column-wise Sums (axis=0):       [12 15 18]
Row-wise Sums (axis=1):          [ 6 15 24]
Row-wise Max Values (axis=1):    [3 6 9]
Index of Max in each row:        [2 2 2]


### 12.2 The Power of `keepdims=True`
When you aggregate an axis, NumPy removes that dimension by default: shape `(3, 4)` becomes `(3,)`.
If you want to subtract row means from the original matrix, the shapes won't align for broadcasting!
Setting `keepdims=True` retains the reduced axis as length 1: shape `(3, 1)`. This allows instant broadcasting!

In [34]:
# Subtract row means from matrix
row_means = np.mean(mat, axis=1, keepdims=True)
print("Row means with keepdims=True shape:", row_means.shape)

# Broadcast subtraction works smoothly!
centered_mat = mat - row_means
print("\nMean-centered rows:\n", centered_mat)

Row means with keepdims=True shape: (3, 1)

Mean-centered rows:
 [[-1.  0.  1.]
 [-1.  0.  1.]
 [-1.  0.  1.]]


### 12.3 NaN-Safe Aggregations
If an array contains even a single `np.nan`, standard functions like `np.mean()` return `NaN`.
NumPy provides `nan`-safe alternatives that ignore missing values: `np.nansum()`, `np.nanmean()`, `np.nanstd()`, `np.nanmedian()`.

In [35]:
messy = np.array([10.0, 20.0, np.nan, 40.0])

print("Standard mean (fails on NaN):", np.mean(messy))
print("NaN-safe mean (ignores NaN): ", np.nanmean(messy))

Standard mean (fails on NaN): nan
NaN-safe mean (ignores NaN):  23.333333333333332


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch13"></a>
---
# Part IV: Linear Algebra, ML & Optimization

## 13. Sorting, Searching & Ranking

### 13.1 `sort()` vs `argsort()`
- **`np.sort(arr)`**: Returns the sorted values.
- **`np.argsort(arr)`**: Returns the **integer indices** that would sort the array.
`argsort` is essential whenever you need to sort multiple aligned arrays simultaneously (e.g., sorting student names based on their exam scores).

In [36]:
students = np.array(['Alice', 'Bob', 'Charlie', 'Diana'])
scores = np.array([78, 95, 62, 88])

# argsort gives indices of lowest to highest scores
sort_order = np.argsort(scores)
print("Score sort order indices:", sort_order)

# Reorder both arrays from highest to lowest score
descending_order = sort_order[::-1]
print("\nRanked Leaderboard:")
for rank, idx in enumerate(descending_order, start=1):
    print(f"Rank {rank}: {students[idx]} with score {scores[idx]}")

Score sort order indices: [2 0 3 1]

Ranked Leaderboard:
Rank 1: Bob with score 95
Rank 2: Diana with score 88
Rank 3: Alice with score 78
Rank 4: Charlie with score 62


### 13.2 Fast $O(N)$ Top-K Selection: `argpartition`
Sorting an array of 1,000,000 elements takes $O(N \log N)$ time.
If you only need the **top 5 largest elements**, full sorting is wasteful.
`np.argpartition(arr, -k)` finds the top $k$ elements in blazing fast **linear time $O(N)$**!

In [37]:
data_pool = np.array([45, 12, 89, 34, 99, 23, 77, 85, 10, 64])

# Find indices of top 3 largest elements without sorting entire array
top3_indices = np.argpartition(data_pool, -3)[-3:]
print("Top 3 values:", data_pool[top3_indices])

Top 3 values: [85 89 99]


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch14"></a>
---
## 14. Linear Algebra Made Intuitive (`np.linalg`)

Linear algebra is the mathematical language of Machine Learning, Neural Networks, and Computer Vision.

### 14.1 Element-Wise Multiplication (`*`) vs Matrix Product (`@`)
- **`A * B`**: Multiplies elements at matching positions.
- **`A @ B`** or **`np.matmul(A, B)`**: Performs real matrix multiplication (dot product of rows and columns).

In [38]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[2, 0], [1, 2]])

print("Element-wise multiplication (A * B):\n", A * B)
print("\nMatrix multiplication (A @ B):\n", A @ B)

Element-wise multiplication (A * B):
 [[2 0]
 [3 8]]

Matrix multiplication (A @ B):
 [[ 4  4]
 [10  8]]


### 14.2 Vector Norms: Measuring Vector Magnitude
- **L1 Norm (Manhattan distance):** Sum of absolute values $\sum |x_i|$.
- **L2 Norm (Euclidean distance):** Straight-line geometric distance $\sqrt{\sum x_i^2}$.

In [39]:
v = np.array([3.0, -4.0])

norm_l1 = np.linalg.norm(v, ord=1)
norm_l2 = np.linalg.norm(v, ord=2)

print("Vector:", v)
print(f"L1 Norm (Manhattan, |3| + |-4|):     {norm_l1}")
print(f"L2 Norm (Euclidean, sqrt(3^2 + 4^2)): {norm_l2}")

Vector: [ 3. -4.]
L1 Norm (Manhattan, |3| + |-4|):     7.0
L2 Norm (Euclidean, sqrt(3^2 + 4^2)): 5.0


### 14.3 Solving Linear Equation Systems ($Ax = b$)
Suppose you want to solve this system of linear equations:
$$2x + y = 8$$
$$x + 3y = 14$$

In matrix form: $Ax = b$.
Never calculate $x = A^{-1}b$ directly (inverting matrices is numerically unstable and slow). Always use **`np.linalg.solve(A, b)`**!

In [40]:
# Coefficient matrix A and target vector b
A = np.array([[2, 1], [1, 3]])
b = np.array([8, 14])

# Solve for x
solution = np.linalg.solve(A, b)
print(f"Solution: x = {solution[0]}, y = {solution[1]}")

Solution: x = 2.0, y = 4.0


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch15"></a>
---
## 15. Practical Machine Learning Vectorization

Here we look at essential Machine Learning building blocks written in pure, vectorized NumPy without any Python `for` loops.

### 15.1 Pairwise Euclidean Distance Matrix
Given $N$ points in $D$ dimensions, calculate the distance between every pair of points.
Using broadcasting and the algebraic identity:
$$\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2(a \cdot b)$$

In [41]:
# 4 points in 2D space
points = np.array([
    [0.0, 0.0],
    [3.0, 0.0],
    [0.0, 4.0],
    [3.0, 4.0]
])

# Sum of squares along rows: shape (4, 1)
sq_norms = np.sum(points ** 2, axis=1, keepdims=True)

# Pairwise squared distances via broadcasting
dists_sq = sq_norms + sq_norms.T - 2 * (points @ points.T)
dists_sq = np.maximum(dists_sq, 0.0) # Guard against tiny floating point precision artifacts
distance_matrix = np.sqrt(dists_sq)

print("Pairwise Euclidean Distance Matrix (4x4):\n", distance_matrix.round(2))

Pairwise Euclidean Distance Matrix (4x4):
 [[0. 3. 4. 5.]
 [3. 0. 5. 4.]
 [4. 5. 0. 3.]
 [5. 4. 3. 0.]]


### 15.2 Numerically Stable Softmax
Softmax turns raw model scores (logits) into class probabilities summing to 1:
$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$
If $z_i = 1000$, $e^{1000}$ triggers floating-point overflow (`inf`), giving `NaN`!
**The Stable Trick:** Subtract the row maximum $\max(z)$ before exponentiating:

In [42]:
def stable_softmax(logits):
    # Subtract row max for stability
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

# Test with huge logits that would normally crash standard softmax
raw_logits = np.array([
    [1000.0, 1001.0, 1002.0],
    [1.0, 2.0, 3.0]
])

probabilities = stable_softmax(raw_logits)
print("Softmax Probabilities:\n", probabilities.round(4))
print("Row sums verify to 1.0:", np.sum(probabilities, axis=1))

Softmax Probabilities:
 [[0.09   0.2447 0.6652]
 [0.09   0.2447 0.6652]]
Row sums verify to 1.0: [1. 1.]


### 15.3 Vectorized One-Hot Encoding
Convert category integer indices into binary vectors in one single line of code using `np.eye`:

In [43]:
labels = np.array([0, 2, 1, 0, 3])
num_classes = 4

# One-hot encoding via identity matrix indexing!
one_hot_matrix = np.eye(num_classes)[labels]

print("Integer Labels:   ", labels)
print("One-Hot Encoded Matrix:\n", one_hot_matrix)

Integer Labels:    [0 2 1 0 3]
One-Hot Encoded Matrix:
 [[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]]


### 15.4 Mean Squared Error (MSE) Loss
The standard loss function for regression:
$$\text{MSE} = \frac{1}{N} \sum_{i=1}^N (y_{\text{true}} - y_{\text{pred}})^2$$

In [44]:
y_true = np.array([10.5, 14.2, 18.0, 22.1, 30.5])
y_pred = np.array([11.0, 13.8, 18.5, 21.0, 31.0])

mse_loss = np.mean((y_true - y_pred) ** 2)
rmse_loss = np.sqrt(mse_loss)

print(f"Mean Squared Error (MSE):  {mse_loss:.4f}")
print(f"Root Mean Squared (RMSE):  {rmse_loss:.4f}")

Mean Squared Error (MSE):  0.4240
Root Mean Squared (RMSE):  0.6512


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch16"></a>
---
## 16. Performance Optimization & Anti-Patterns

### 16.1 The `#1` NumPy Anti-Pattern: `np.append()` Inside a Loop
In Python lists, `list.append()` takes $O(1)$ amortized time.
In NumPy, `np.append()` **allocates a brand-new array and copies all existing data into it!**
Running `np.append()` inside a loop of $N$ iterations runs in **$O(N^2)$ time** and destroys performance.

**The Fix:** Always **preallocate** an empty array of the final size using `np.empty()` or `np.zeros()`!

In [45]:
N = 25_000

# ❌ The Anti-Pattern: np.append inside loop
t0 = time.perf_counter()
bad_arr = np.array([])
for i in range(N):
    bad_arr = np.append(bad_arr, i)
t1 = time.perf_counter()
time_bad = t1 - t0

# ✅ The Best Practice: Preallocate memory once!
t0 = time.perf_counter()
good_arr = np.empty(N, dtype=np.float64)
for i in range(N):
    good_arr[i] = i
t1 = time.perf_counter()
time_good = t1 - t0

print(f"❌ np.append loop:   {time_bad:.4f} seconds")
print(f"✅ Preallocation:     {time_good:.4f} seconds")
print(f"⚡ Preallocation is  {time_bad / time_good:.1f}x faster!")

❌ np.append loop:   0.1169 seconds
✅ Preallocation:     0.0022 seconds
⚡ Preallocation is  53.3x faster!


### 16.2 In-Place Operations to Save Memory
Expressions like `a = a + b` allocate a temporary intermediate array before reassigning.
Using in-place operators (`+=`, `*=`) modifies data directly in RAM without allocating new memory:

In [46]:
big_a = np.ones((1000, 1000), dtype=np.float64)
big_b = np.ones((1000, 1000), dtype=np.float64)

# Memory address before
id_before = id(big_a)

# In-place operation modifies existing buffer in RAM
big_a += big_b
id_after = id(big_a)

print("Same memory buffer maintained?", id_before == id_after)

Same memory buffer maintained? True


---
<a id="summary"></a>
## 🏆 Comprehensive NumPy Quick Reference Cheat Sheet

| Category | High-Frequency Function | Quick Example |
| :--- | :--- | :--- |
| **Creation** | `np.array`, `np.zeros`, `np.ones`, `np.full`, `np.arange`, `np.linspace`, `np.eye` | `np.linspace(0, 1, 10)` |
| **Inspection** | `arr.shape`, `arr.ndim`, `arr.dtype`, `arr.size`, `arr.nbytes` | `arr.shape` -> `(3, 4)` |
| **Reshaping** | `arr.reshape(-1)`, `arr.ravel()`, `arr.flatten()`, `arr.T`, `np.squeeze` | `arr.reshape(2, -1)` |
| **Slicing** | `arr[start:stop:step]`, `arr[row_slice, col_slice]` | `arr[1:3, :]` |
| **Filtering** | `arr[condition]`, `np.where(cond, x, y)`, `&`, `\|`, `~` | `np.where(x > 0, x, 0)` |
| **Math** | `np.exp`, `np.log`, `np.sqrt`, `np.sin`, `np.abs`, `np.clip` | `np.clip(arr, 0, 1)` |
| **Aggregations** | `arr.sum(axis=...)`, `arr.mean()`, `arr.std()`, `arr.argmax()`, `keepdims=True` | `np.mean(mat, axis=0)` |
| **Broadcasting** | Automatic dimension expansion from right to left | `(3, 4) + (4,) -> (3, 4)` |
| **Stacking** | `np.vstack`, `np.hstack`, `np.concatenate(axis=...)` | `np.vstack([a, b])` |
| **Linear Algebra** | `A @ B`, `np.linalg.solve(A, b)`, `np.linalg.norm(v)` | `A @ B` |
| **Random (`rng`)** | `rng.random()`, `rng.integers()`, `rng.normal()`, `rng.choice()`, `rng.shuffle()`| `rng.normal(0, 1, 10)` |

---
### 🎯 Next Steps: Put Knowledge Into Practice!
Deepen your understanding with our companion practice notebooks:
- 🏋️ [02_numpy_exercise_1.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/02_numpy_exercise_1.ipynb): 20 Hands-on exercises covering Array Creation, Slicing, and Reshaping.
- 💡 [03_numpy_exercise_1_solutions.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/03_numpy_exercise_1_solutions.ipynb): Complete worked solutions for Exercise 1.
- 🚀 [04_numpy_exercise_2.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/04_numpy_exercise_2.ipynb): 15 Advanced problems covering Linear Algebra and ML Vectorization.
- 💡 [05_numpy_exercise_2_solutions.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/numpy/05_numpy_exercise_2_solutions.ipynb): Detailed solutions and vectorization derivations for Exercise 2.